# Olist AI Platform: Data Exploration & Profiling

## Objective

This notebook performs the initial exploration and profiling of the Olist Brazilian E-Commerce dataset.

The objectives are to:

- Understand the structure and content of the raw datasets
- Identify tables, columns, data types, and relationships
- Analyze missing values and duplicates
- Identify primary and foreign keys
- Analyze date and timestamp fields
- Validate potential business rules
- Understand the data required for analytics and ML use cases
- Prepare the design of the Bronze, Silver, and Gold data layers

This notebook is exploratory only. Production data pipelines will be implemented separately under `pipelines/`.

# 1. Imports & Configuration

In [5]:
from pathlib import Path 
import pandas as pd
import numpy as np

In [6]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 2. Define Project Paths

The raw Olist CSV files are stored under `data/raw/`.

At this stage, the data is kept in its original form and will not be modified.

In [7]:
PROJECT_ROOT = Path("..")

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"

print(f"Raw data path: {RAW_DATA_PATH.resolve()}")

Raw data path: C:\Users\user\Documents\projects\olist-ai-platform\data\raw


In [8]:
assert RAW_DATA_PATH.exists(), f"Raw data directory not found: {RAW_DATA_PATH}"

print("Raw data directory exists.")

Raw data directory exists.


# 3. Discover Raw Dataset Files

In [9]:
files = sorted(RAW_DATA_PATH.glob("*.csv"))

print(f"Number of CSV files found: {len(files)}\n")

for file in files:
    print(file.name)

Number of CSV files found: 9

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


# 4. Load the Raw Datasets

For this initial exploration, Pandas is sufficient because the Olist dataset is relatively small.

Later, Spark will be used for the production ingestion and transformation pipelines.

In [10]:
datasets = {}

for file in files:
    dataset_name = (
        file.stem
        .replace("olist_", "")
        .replace("_dataset", "")
    )
    
    datasets[dataset_name] = pd.read_csv(file)

print("Datasets loaded successfully.")

Datasets loaded successfully.


# 5. Dataset Overview

We first inspect the size of each dataset to understand the overall data landscape.

In [11]:
dataset_overview = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "memory_mb": df.memory_usage(deep=True).sum() / 1024**2,
        }
        for name, df in datasets.items()
    ]
).sort_values("rows", ascending=False)

dataset_overview

,dataset,rows,columns,memory_mb
1,geolocation,1000163,5,50.12
2,order_items,112650,7,18.37
3,order_payments,103886,5,8.11
0,customers,99441,5,11.03
5,orders,99441,8,21.95
4,order_reviews,99224,7,17.84
6,products,32951,9,3.73
7,sellers,3095,4,0.22
8,product_category_name_translation,71,2,0.00


# 6. Inspect Dataset Schemas

We inspect the columns and data types of every raw dataset.

Understanding the raw schema is necessary before designing the Bronze and Silver layers.

In [12]:
for name, df in datasets.items():
    print("=" * 80)
    print(f"DATASET: {name}")
    print("=" * 80)
    print(df.info())
    print()

DATASET: customers
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB
None

DATASET: geolocation
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000

# 7. Column-Level Metadata

We create a centralized metadata table containing:

- Dataset name
- Column name
- Data type
- Number of missing values
- Missing-value percentage
- Number of unique values

In [14]:
column_metadata = []

for name, df in datasets.items():
    for column in df.columns:
        column_metadata.append(
            {
                "dataset": name,
                "column": column,
                "dtype": str(df[column].dtype),
                "missing_count": df[column].isna().sum(),
                "missing_pct": df[column].isna().mean() * 100,
                "unique_count": df[column].nunique(dropna=True),
            }
        )

column_metadata = pd.DataFrame(column_metadata)

#column_metadata.head(20)
print(column_metadata)

                              dataset                         column    dtype  \
0                           customers                    customer_id      str   
1                           customers             customer_unique_id      str   
2                           customers       customer_zip_code_prefix    int64   
3                           customers                  customer_city      str   
4                           customers                 customer_state      str   
5                         geolocation    geolocation_zip_code_prefix    int64   
6                         geolocation                geolocation_lat  float64   
7                         geolocation                geolocation_lng  float64   
8                         geolocation               geolocation_city      str   
9                         geolocation              geolocation_state      str   
10                        order_items                       order_id      str   
11                        or

# 8. Missing Values Analysis

Missing values are analyzed before defining transformation rules.

A missing value does not necessarily represent a data-quality problem. For example, delivery-related fields may naturally be missing for orders that have not yet been delivered.

In [15]:
missing_summary = (
    column_metadata[
        column_metadata["missing_count"] > 0
    ]
    .sort_values(
        ["dataset", "missing_pct"],
        ascending=[True, False]
    )
)

missing_summary

,dataset,column,dtype,missing_count,missing_pct,unique_count
25,order_reviews,review_comment_title,str,87656,88.34,4527
26,order_reviews,review_comment_message,str,58247,58.70,36159
35,orders,order_delivered_customer_date,str,2965,2.98,95664
34,orders,order_delivered_carrier_date,str,1783,1.79,81018
33,orders,order_approved_at,str,160,0.16,90733
38,products,product_category_name,str,610,1.85,73
39,products,product_name_lenght,float64,610,1.85,66
40,products,product_description_lenght,float64,610,1.85,2960
41,products,product_photos_qty,float64,610,1.85,19
42,products,product_weight_g,float64,2,0.01,2204


# 9. Duplicate Analysis

We check for fully duplicated rows in each dataset.

This is different from checking whether a primary key is unique.

In [16]:
duplicate_summary = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": len(df),
            "duplicate_rows": df.duplicated().sum(),
            "duplicate_pct": df.duplicated().mean() * 100,
        }
        for name, df in datasets.items()
    ]
)

duplicate_summary

,dataset,rows,duplicate_rows,duplicate_pct
0,customers,99441,0,0.00
1,geolocation,1000163,261831,26.18
2,order_items,112650,0,0.00
3,order_payments,103886,0,0.00
4,order_reviews,99224,0,0.00
5,orders,99441,0,0.00
6,products,32951,0,0.00
7,sellers,3095,0,0.00
8,product_category_name_translation,71,0,0.00


# 10. Preview Each Dataset

In [17]:
for name, df in datasets.items():
    print("=" * 80)
    print(f"{name.upper()}")
    display(df.head(3))

CUSTOMERS


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


GEOLOCATION


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP


ORDER_ITEMS


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,"1,000.00",30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00


SELLERS


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


PRODUCT_CATEGORY_NAME_TRANSLATION


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


# 11. Orders Dataset — Central Entity

The `orders` table is one of the central entities of the Olist data model.

It contains the order lifecycle and several timestamps that will later be used for:

- Delivery analytics
- Delivery delay calculation
- Seller performance features
- Delivery risk prediction

In [18]:
orders = datasets["orders"]
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### 11.1 Orders Schema

In [19]:
orders.info()
orders.columns.tolist()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB


['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

### 11.2 Order Status Distribution

In [22]:
orders["order_status"].value_counts(dropna=False)


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [23]:
orders["order_status"].value_counts(normalize=True, dropna=False) * 100

order_status
delivered     97.02
shipped        1.11
canceled       0.63
unavailable    0.61
invoiced       0.32
processing     0.30
created        0.01
approved       0.00
Name: proportion, dtype: float64

# 12. Customer Dataset

In [27]:
customers = datasets["customers"]
customers.info()
customers.head()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


# 13. Seller Dataset

In [28]:
sellers = datasets["sellers"]
sellers.info()
sellers.head()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 230.3 KB


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [29]:
print("Rows:", len(sellers))
print("Unique sellers:", sellers["seller_id"].nunique())

Rows: 3095
Unique sellers: 3095


# 14. Product Dataset

In [30]:
products = datasets["products"]
products.info()
products.head()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 3.7 MB


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,"1,000.00",30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.00,261.00,1.00,371.00,26.00,4.00,26.00
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.00,402.00,4.00,625.00,20.00,17.00,13.00


In [31]:
products["product_id"].nunique(), len(products)

(32951, 32951)

# 15. Order Items Dataset

The order-items table represents the products purchased within each order.

An order can contain multiple items and therefore `order_id` is not expected to be unique here.

In [32]:
order_items = datasets["order_items"]
order_items.info()
order_items.head()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 18.4 MB


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [33]:
print("Rows:", len(order_items))
print("Unique orders:", order_items["order_id"].nunique())
print("Unique products:", order_items["product_id"].nunique())
print("Unique sellers:", order_items["seller_id"].nunique())

Rows: 112650
Unique orders: 98666
Unique products: 32951
Unique sellers: 3095


# 16. Order Payments

In [34]:
payments = datasets["order_payments"]
payments.info()
payments.head()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 8.1 MB


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [35]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

# 17. Order Reviews

In [36]:
reviews = datasets["order_reviews"]
reviews.info()
reviews.head()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 17.8 MB


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [37]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

# 18. Geolocation Dataset

In [38]:
geolocation = datasets["geolocation"]
geolocation.info()
geolocation.head()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 50.1 MB


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP
3,1041,-23.54,-46.64,sao paulo,SP
4,1035,-23.54,-46.64,sao paulo,SP


In [39]:
geolocation.duplicated().sum()

np.int64(261831)

# 19. Product Category Translation

In [40]:
category_translation = datasets["product_category_name_translation"]
category_translation.info()
category_translation.head()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 3.5 KB


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


# 20. Date & Timestamp Analysis

The order lifecycle contains several important timestamps.

These fields will later support:

- Delivery-time analytics
- Late-delivery calculation
- Seller historical performance
- Delivery-risk modeling

At this stage we only inspect the raw values.

In [41]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders[date_columns].head()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [42]:
orders_exploration = orders.copy()

for column in date_columns:
    orders_exploration[column] = pd.to_datetime(
        orders_exploration[column],
        errors="coerce"
    )

orders_exploration[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

# 21. Order Timeline Analysis

In [44]:
orders_exploration[date_columns].describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99281,97658,96476,99441
mean,2017-12-31 08:43:12.776581,2017-12-31 18:35:24.098800,2018-01-04 21:49:48.138278,2018-01-14 12:09:19.035542,2018-01-24 03:08:37.730111
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00
25%,2017-09-12 14:46:19,2017-09-12 23:24:16,2017-09-15 22:28:50.250000,2017-09-25 22:07:22.250000,2017-10-03 00:00:00
50%,2018-01-18 23:04:36,2018-01-19 11:36:13,2018-01-24 16:10:58,2018-02-02 19:28:10.500000,2018-02-15 00:00:00
75%,2018-05-04 15:42:16,2018-05-04 20:35:10,2018-05-08 13:37:45,2018-05-15 22:48:52.250000,2018-05-25 00:00:00
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12 00:00:00


# 22. Delivery Delay — Exploratory Analysis

The dataset provides both:

- Actual customer delivery date
- Estimated delivery date

This allows us to derive a delivery-delay indicator.

This feature will later be implemented in the Silver/Gold transformation pipeline rather than in this notebook.

In [45]:
orders_exploration["delivery_delay_days"] = (
    orders_exploration["order_delivered_customer_date"]
    - orders_exploration["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 3600)

In [46]:
orders_exploration[
    ["delivery_delay_days"]
].describe()

,delivery_delay_days
count,"96,476.00"
mean,-11.18
std,10.19
min,-146.02
25%,-16.24
50%,-11.95
75%,-6.39
max,188.98


In [47]:
orders_exploration["is_late"] = (
    orders_exploration["delivery_delay_days"] > 0
)

In [48]:
orders_exploration["is_late"].value_counts(dropna=False)

is_late
False    91614
True      7827
Name: count, dtype: int64

In [49]:
orders_exploration["is_late"].value_counts(
    normalize=True,
    dropna=False
) * 100

is_late
False   92.13
True     7.87
Name: proportion, dtype: float64

# 23. Seller Performance — Initial Exploration

Seller performance will later be used as a feature for delivery-risk prediction.

The raw dataset does not directly contain a field called `historical_seller_performance`.

Instead, we can derive historical seller performance from previous orders.

In [50]:
seller_orders = order_items[
    ["order_id", "seller_id"]
].drop_duplicates().merge(
    orders_exploration[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_delay_days",
            "is_late",
        ]
    ],
    on="order_id",
    how="left",
)

In [51]:
seller_orders.groupby("seller_id").agg(
    orders=("order_id", "nunique"),
    late_rate=("is_late", "mean"),
    avg_delay_days=("delivery_delay_days", "mean"),
).sort_values(
    "late_rate",
    ascending=False
).head(20)

,orders,late_rate,avg_delay_days
seller_id,,,
be1e9e378700cecaa4ebf71433d7915c,2,1.00,24.26
bc8c8d665ec4664d286be0d521722b19,1,1.00,4.66
05ca864204d09595ae591b93ea9cf93d,1,1.00,0.56
2a50b7ee5aebecc6fd0ff9784a4747d6,1,1.00,17.64
20f0aeea30bc3b8c4420be8ced4226c0,1,1.00,13.52
63704069d9bd3a75c1cf59babe56004a,1,1.00,7.78
6524b847b982cd56bb5d4b02b776ee42,1,1.00,16.82
c13ef0cfbe42f190780f621ce81f2234,1,1.00,6.63
2a73cba571d90c694b7caca072ccf6ce,1,1.00,0.72


# 24. Data Relationships

We now investigate the relationships between the main entities.

In [52]:
orders["customer_id"].nunique()

99441

In [53]:
customers["customer_id"].nunique()

99441

In [54]:
order_items["order_id"].nunique()

98666

In [55]:
orders["order_id"].nunique()

99441

In [56]:
order_items["product_id"].nunique()

32951

In [57]:
products["product_id"].nunique()

32951

In [58]:
order_items["seller_id"].nunique()

3095

In [59]:
sellers["seller_id"].nunique()

3095

# 25. Referential Integrity Checks

We verify whether foreign-key values in transactional tables exist in their corresponding dimension/entity tables.

In [60]:
missing_customer_ids = set(orders["customer_id"]) - set(customers["customer_id"])

print(f"Customer IDs missing from customer table: {len(missing_customer_ids)}")

Customer IDs missing from customer table: 0


In [61]:
missing_product_ids = set(order_items["product_id"]) - set(products["product_id"])

print(f"Product IDs missing from product table: {len(missing_product_ids)}")

Product IDs missing from product table: 0


In [62]:
missing_seller_ids = set(order_items["seller_id"]) - set(sellers["seller_id"])

print(f"Seller IDs missing from seller table: {len(missing_seller_ids)}")

Seller IDs missing from seller table: 0


# 26. Business Rules — Initial Validation

We inspect several potential business rules that will later become data-quality checks.

In [63]:
#Rule 1: Delivery after purchase

valid_delivery_dates = orders_exploration[
    orders_exploration["order_delivered_customer_date"].notna()
]

invalid_delivery_sequence = valid_delivery_dates[
    valid_delivery_dates["order_delivered_customer_date"]
    < valid_delivery_dates["order_purchase_timestamp"]
]

print(
    f"Orders delivered before purchase: "
    f"{len(invalid_delivery_sequence)}"
)

Orders delivered before purchase: 0


In [64]:
#Rule 2: Estimated delivery dates

orders_exploration[
    [
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].isna().sum()

order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

# 27. Initial Business Questions

Based on the dataset, the platform should eventually answer questions such as:

### Sales
- How many orders are placed?
- What is the total revenue?
- What is the average order value?
- Which product categories generate the most revenue?

### Customers
- Who are the most valuable customers?
- Which regions generate the most orders?
- What is the customer repeat-purchase behavior?

### Sellers
- Which sellers have the highest sales?
- Which sellers have the highest late-delivery rate?
- Which sellers consistently perform well?

### Delivery
- What percentage of orders are delivered late?
- Which states have the highest delivery delays?
- Which sellers have elevated delivery risk?

### ML
- Can we predict whether an order will be delivered late?
- Which features contribute most to delivery risk?

# 28. Initial Data Model

Based on the exploration, the main relationships can be summarized as:

```text
Customers
    │
    │ customer_id
    ▼
Orders
    │
    │ order_id
    ▼
Order Items
    │
    ├──────────► Products
    │
    └──────────► Sellers

Orders
    ├──────────► Payments
    │
    └──────────► Reviews


---

# 29. Candidate Warehouse Model / Candidate Analytical Model

The exploration suggests a dimensional model for the analytical warehouse.

Potential dimensions:

- `dim_customer`
- `dim_seller`
- `dim_product`
- `dim_date`
- `dim_geolocation`

Potential facts:

- `fact_orders`
- `fact_order_items`
- `fact_payments`
- `fact_reviews`

Potential data marts:

- `delivery_mart`
- `sales_mart`
- `customer_mart`
- `seller_mart`

This model will be finalized during the data architecture implementation phase.

# 30. Data Exploration Conclusions

## Main findings

### Data sources

The Olist dataset contains multiple interconnected datasets covering:

- Customers
- Orders
- Order items
- Products
- Sellers
- Payments
- Reviews
- Geolocation
- Product categories

### Important observations

- `orders` is a central transactional entity.
- `order_items` creates the relationship between orders, products, and sellers.
- Several timestamp fields enable delivery-performance analysis.
- Estimated and actual delivery dates can be used to derive delivery-delay metrics.
- Seller historical performance can be derived from previous orders rather than being directly available as a raw feature.
- Missing values need to be interpreted according to business context rather than automatically removed.

### Next steps

The next stage is to implement the production data pipeline:

```text
Raw CSV
   ↓
Spark ingestion
   ↓
Bronze Parquet
   ↓
Silver transformation
   ↓
Gold analytical model
   ↓
DuckDB warehouse